In [ ]:
import osmnx as ox
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import LineString
from pathlib import Path

## Build an FMM-ready edge file from OSMnx

creates a directed edge layer with columns fid, u, v, and geometry, which matches the common FMM pattern

`uv pip install osmnx`

In [ ]:
EDGES_GDF_OUTPUT_PATH = "../outputs/nyc_output_tabular/nyc_edges.gpkg"
output_path = Path(EDGES_GDF_OUTPUT_PATH)
# {"all", "all_public", "bike", "drive", "drive_service", "walk"} What type of street network to retrieve
NETWORK_TYPE = "drive"

In [ ]:
output_path.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
def build_fmm_network_from_place(place: str, out_shp: str) -> None:
    """ Build the road network with OSMnx.

        the exported network needs fid, u, and v columns to be compatible with FMM
    """
    # output_dir_path = Path(out_dir)
    # output_dir_path.parent.mkdir(parents=True, exist_ok=True)

    # Download drivable network
    G = ox.graph_from_place(place, network_type="drive")

    # Convert to GeoDataFrames
    nodes_gdf, edges_gdf = ox.graph_to_gdfs(G)

    # Flatten (u, v, key) index into columns
    edges_gdf = edges_gdf.reset_index()

    # FMM-compatible columns
    edges_gdf["fid"] = np.arange(len(edges_gdf), dtype="int64")

    # Keep required columns first
    keep = ["fid", "u", "v", "geometry"]
    optional = ["osmid", "length", "highway", "name", "oneway"]
    for col in optional:
        if col not in edges_gdf.columns:
            edges_gdf[col] = None

    edges_gdf = edges_gdf[keep + optional].copy()

    # shp_path = os.path.join(out_dir, "edges.shp")
    # shp_path = output_dir_path / "edges.shp"
    edges_gdf.to_file(out_shp)

In [ ]:
# Pick one:
# Better if you want tighter control:
# north, south, east, west = 40.93, 40.47, -73.68, -74.30
# G = ox.graph_from_bbox((north, south, east, west), network_type="drive")
G = ox.graph_from_place("New York City, New York, USA", network_type=NETWORK_TYPE)

In [20]:
# Convert to GeoDataFrames
nodes_gdf, edges_gdf = ox.graph_to_gdfs(G)

# Flatten the multi-index (u, v, key)
edges_gdf = edges_gdf.reset_index()

In [ ]:
# Keep only what we need first
keep_cols = ["u", "v", "geometry"]
# maxspeed in mph
optional_cols = ["osmid", "length", "highway", "name", "oneway"]
for c in optional_cols:
    if c not in edges_gdf.columns:
        edges_gdf[c] = None

edges_gdf = edges_gdf[keep_cols + optional_cols].copy()

maxspeed column has about 97039 null values so not using it as about 70% dont have a max speed.

In [26]:
# preprocessing
# remove white space and 'mph' in maxspeed
# edges_gdf["maxspeed"] = edges_gdf["maxspeed"].str.replace(r"\s+|mph", "", regex=True)

In [27]:
# edges_gdf["maxspeed"].value_counts()

In [ ]:
# edges_gdf["maxspeed"].isnull().sum()

np.int64(97039)

In [ ]:
# edges_gdf["maxspeed"].shape

(139156,)

In [28]:
edges_gdf.head(3)

,u,v,geometry,osmid,length,highway,name,oneway,maxspeed
0,39076461,274283981,"LINESTRING (-73.79475 40.78635, -73.79462 40.7...",25161349,819.501666,motorway,Cross Island Parkway,True,50 mph
1,39076461,42854803,"LINESTRING (-73.79475 40.78635, -73.79332 40.7...",25161578,268.144095,motorway_link,NaN,True,NaN
2,39076490,277672046,"LINESTRING (-73.75709 40.76243, -73.75721 40.7...",5699971,246.749804,motorway_link,NaN,True,NaN


In [ ]:
# Create a unique edge id for FMM
edges_gdf["fid"] = range(len(edges_gdf))

# Reorder columns
edges_gdf = edges_gdf[
    ["fid", "u", "v", "geometry", "osmid", "length", "highway", "name", "oneway"]
]

# Save as GeoPackage (preferred) or shapefile
edges_gdf.to_file(EDGES_GDF_OUTPUT_PATH, layer="edges", driver="GPKG")
# or:
# edges_gdf.to_file("nyc_edges.shp")

In [4]:
for u, v, k, data in G.edges(data=True, keys=True):
    print(f"Edge ({u}, {v}) with key {k} has attributes: {data}")
    break

Edge (39076461, 274283981) with key 0 has attributes: {'osmid': 25161349, 'highway': 'motorway', 'lanes': '2', 'maxspeed': '50 mph', 'name': 'Cross Island Parkway', 'oneway': True, 'ref': 'CI', 'reversed': False, 'length': np.float64(819.5016661477804), 'geometry': <LINESTRING (-73.795 40.786, -73.795 40.786, -73.794 40.786, -73.794 40.786,...>}


## Convert cleaned parquet to FMM GPS CSV

FMM accepts a CSV point file where each row is one observation with trajectory id, longitude, latitude, and optional timestamp; the file must already be sorted by id and timestamp

CSV point file: a CSV file with a header row and columns separated by ;. Each row stores a single observation containing id(integer), x(longitude), y(latitude), timestamp(optional, integer). The file must be sorted already by id and timestamp (trajectory will be passed sequentially). The id, x, y and timestamp column names will be specified by the user.

In [13]:
GPS_TRAJ_PARQUET_PATH = "../data/nyc_output_tabular/output/traj_cleaned.parquet"

GPS_TRAJ_FMM_PATH = "../outputs/nyc_output_tabular/nyc_gps_points_fmm_ready.csv"
output_path = Path(GPS_TRAJ_FMM_PATH)

In [14]:
output_path.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
def build_fmm_points_csv(
    parquet_path: str,
    out_csv: str,
    trip_id_map_csv: str | None = None
) -> pd.DataFrame:
    """
    CSV point file: a CSV file with a header row and columns separated by ;. 
    Each row stores a single observation containing id(integer), x(longitude), y(latitude), timestamp(optional, integer). 
    
    The file must be sorted already by id and timestamp (trajectory will be passed sequentially). The id, x, y and timestamp column names will be specified by the user.

    Since our tid is only unique within a user, build a unique trip key from uid + tid. FMM's docs say id is an integer, so remap each unique trip key to an integer trajectory id.
    """
    df = pd.read_parquet(parquet_path).copy()

    # Unique trip key
    df["trip_key"] = df["uid"].astype(str) + "_" + df["tid"].astype(str)

    # Integer trajectory ids for FMM
    trip_keys = pd.Index(df["trip_key"].unique())
    trip_id_map = pd.DataFrame({
        "trip_key": trip_keys,
        "id": range(len(trip_keys))
    })

    df = df.merge(trip_id_map, on="trip_key", how="left")

    # Integer unix timestamp in seconds
    df["timestamp"] = pd.to_datetime(df["datetime"]).astype("int64") // 10**9

    # Sort exactly as required
    df = df.sort_values(["id", "timestamp"]).reset_index(drop=True)

    gps_df = pd.DataFrame({
        "id": df["id"].astype("int64"),
        "x": df["lng"].astype(float),
        "y": df["lat"].astype(float),
        "timestamp": df["timestamp"].astype("int64"),
    })

    gps_df.to_csv(out_csv, sep=";", index=False)

    if trip_id_map_csv is not None:
        trip_id_map.to_csv(trip_id_map_csv, index=False)

    return trip_id_map

`uv pip install pyArrow`

In [15]:
df = pd.read_parquet(GPS_TRAJ_PARQUET_PATH).copy()

# Build one trajectory id per trip
df["trip_id"] = df["uid"].astype(str) + "_" + df["tid"].astype(str)

# Sort exactly as FMM expects
df = df.sort_values(["trip_id", "datetime"]).reset_index(drop=True)

# FMM point CSV wants longitude=x, latitude=y
gps_df = pd.DataFrame({
    "id": df["trip_id"],
    "x": df["lng"],
    "y": df["lat"],
    "timestamp": pd.to_datetime(df["datetime"]).astype("int64") // 10**9
})

# IMPORTANT: FMM point CSV uses semicolon separator
gps_df.to_csv(GPS_TRAJ_FMM_PATH, sep=";", index=False)

In [22]:
gps_df.describe()

,x,y,timestamp
count,6.625802e+06,6.625802e+06,6.625802e+06
mean,-7.406700e+01,4.070139e+01,1.668448e+09
std,8.884941e-02,9.993298e-02,4.029098e+07
min,-7.425900e+01,4.047704e+01,1.203292e+09
25%,-7.414831e+01,4.061133e+01,1.660215e+09
50%,-7.405205e+01,4.069093e+01,1.672397e+09
75%,-7.399015e+01,4.077322e+01,1.684830e+09
max,-7.389900e+01,4.091800e+01,1.728208e+09


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6625802 entries, 0 to 6625801
Data columns (total 6 columns):
 #   Column    Dtype         
---  ------    -----         
 0   datetime  datetime64[ns]
 1   uid       int64         
 2   tid       str           
 3   lat       float64       
 4   lng       float64       
 5   trip_id   str           
dtypes: datetime64[ns](1), float64(2), int64(1), str(2)
memory usage: 409.7 MB


In [8]:
df.isnull().sum()

datetime    0
uid         0
tid         0
lat         0
lng         0
trip_id     0
dtype: int64

## Create road network and prepare trajectory to be used by FMM

In [ ]:
BASE_DATA_DIR = "../data/nyc_output_tabular/output"
BASE_OUTPUT_DIR = "../outputs/nyc_output_tabular"

PLACE = "New York City, New York, USA"
ROAD_NETWORK_OUT_DIR = f"{BASE_OUTPUT_DIR}/fmm_nyc.shp"
road_network_out_dir_path = Path(ROAD_NETWORK_OUT_DIR)
road_network_out_dir_path.parent.mkdir(parents=True, exist_ok=True)

GPS_TRAJ_PARQUET_PATH = f"{BASE_DATA_DIR}/traj_cleaned.parquet"
GPS_TRAJ_FMM_OUTPUT_PATH = f"{BASE_OUTPUT_DIR}/nyc_gps_points_fmm_ready.csv"
TRIP_ID_MAP_CSV = f"{BASE_OUTPUT_DIR}/nyc_gps_points_fmm_trip_id_map.csv"
# {"all", "all_public", "bike", "drive", "drive_service", "walk"} What type of street network to retrieve
NETWORK_TYPE = "drive"
gps_traj_fmm_output_path = Path(GPS_TRAJ_FMM_OUTPUT_PATH)
gps_traj_fmm_output_path.parent.mkdir(parents=True, exist_ok=True)



build_fmm_network_from_place(PLACE, ROAD_NETWORK_OUT_DIR)

trip_map = build_fmm_points_csv(
    parquet_path=GPS_TRAJ_PARQUET_PATH,
    out_csv=GPS_TRAJ_FMM_OUTPUT_PATH,
    trip_id_map_csv=TRIP_ID_MAP_CSV,
)

## FMM Code

test text

```bash
sudo apt-get install -y \
    make \
    cmake \
    libboost-dev \
    libboost-serialization-dev \
    libgdal-dev \
    gdal-bin \
    libbz2-dev \
    libexpat1-dev \
    swig \
    python3-dev
```


It will build executable files under the build folder, which are installed to /usr/local/bin
```bash
git clone https://github.com/cyang-kth/fmm.git
cd fmm

mkdir build
cd build

cmake ..
make -j$(nproc)
sudo make install
```

